In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import pickle
import aux_functions as aux
importlib.reload(aux)

In [ ]:
MONKEY = 'M08'
# SESSION = 'm08cat156'
SESSION = 'm08cat255'
STR_DETAILS = f'Session {SESSION}'

In [ ]:
with open(f"session_data/{SESSION}.pkl", "rb") as file:
    df_session= pickle.load(file)

df_session[df_session['RF_type']=='Focal Foveal']
STR_DETAILS += ' Foveal'

df_session

#### Dataframe com uma linha a cada (trial,sacada,neuronio)

In [ ]:
path_destiny = f'dados/{MONKEY}_neurons/neurons sorted' # linux
list_neuron_ids = df_session['search']
list_prefered_cat = df_session['prefer']
list_brain_area = df_session['brain_area']

df_trials, dicio_firings = aux.get_df_trials_all_saccs_foveal(list_neuron_ids, path_destiny, list_prefered_cat, list_brain_area)
df_trials

#### Gerar dataframe de Y+Bins que usaremos nos modelos de ML

In [ ]:
start_time = -300
end_time = 300
bin_width = 50
step = 50
t_ends = np.arange(start_time + bin_width, end_time + step, step)
t_ends

In [ ]:
decoding_type = 'CATEGORY'
decoding_type = 'CHANGE_OF_CATEGORY'

In [ ]:
t_starts = t_ends - bin_width  

neurons_order = list(df_trials['neuron'].unique())
saccades_order = list(df_trials['trial+sacc'].unique())

# guardar cada fix time
df_indexed = df_trials.set_index(['trial+sacc', 'neuron'])
dicio_fix_times = df_indexed['fix_time'].to_dict()
del df_indexed

# guardar a variável que queremos decodificar
df_saccades = df_trials.drop_duplicates('trial+sacc').set_index('trial+sacc')
if decoding_type == 'CHANGE_OF_CATEGORY':
    dicio_Y = (df_saccades['sacc_img_cath'] == df_saccades['img_cath_anterior']).astype(int).to_dict()
elif decoding_type == 'CATEGORY':
    dicio_Y = df_saccades['sacc_img_cath'].to_dict()
del df_saccades

dicio_firings_np = {
    neuron: np.sort(np.array(spikes) * 1000.0) 
    for neuron, spikes in dicio_firings.items()
}

data_trials_run_model = []
for saccade_id in saccades_order:
    data_model_row = {'Y': dicio_Y.get(saccade_id, 0)}
    for neuron in neurons_order:
        fix_time = dicio_fix_times.get((saccade_id, neuron))
        # if pd.isna(fix_time):
        #     for i in range(len(t_ends)):
        #         data_model_row[f'{neuron}_Bin{i}'] = 0.0
        #     continue
        fix_time_ms = fix_time * 1000.0

        session_spikes = dicio_firings_np[neuron][0]
        window_start_global = fix_time_ms + start_time
        window_end_global = fix_time_ms + end_time
        
        idx_start = np.searchsorted(session_spikes, window_start_global)
        idx_end = np.searchsorted(session_spikes, window_end_global)
        
        trial_spikes = session_spikes[idx_start:idx_end] - fix_time_ms
        
        for i, (t_s, t_e) in enumerate(zip(t_starts, t_ends)):
            count = np.sum((trial_spikes >= t_s) & (trial_spikes < t_e))
            fr = count / (bin_width / 1000.0)
            data_model_row[f'{neuron}_Bin{i}'] = fr
            
    data_trials_run_model.append(data_model_row)
df_run_model = pd.DataFrame(data_trials_run_model)

#### rodar decoding

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef

def zscore_per_neuron(X_train, X_test):
    X_train_z = X_train.copy()
    X_test_z = X_test.copy()
    neuron_ids = set([col.split('_Bin')[0] for col in X_train.columns if '_Bin' in col])
    
    for nid in neuron_ids:
        cols_neuron = [col for col in X_train.columns if col.startswith(f"{nid}_Bin")]
        train_values = X_train[cols_neuron].values.flatten()
        
        mu = np.mean(train_values)
        sigma = np.std(train_values) if np.std(train_values) > 0 else 1e-8
            
        X_train_z[cols_neuron] = (X_train[cols_neuron] - mu) / sigma
        X_test_z[cols_neuron] = (X_test[cols_neuron] - mu) / sigma
        
    return X_train_z, X_test_z


def run_temporal_decoding_generic(df_run_model, start_time, bin_width, step, window_size_bins, num_folds=5, verbose=True):

    neuron_ids = list(set([col.split('_Bin')[0] for col in df_run_model.columns if '_Bin' in col]))
    total_bins = len([col for col in df_run_model.columns if col.startswith(f"{neuron_ids[0]}_Bin")])
    
    # Quantos "passos" cabem dentro de um bin? (ex: bin de 50ms, passo de 10ms -> offset = 5)
    offset = bin_width // step
    
    # Se window_size_bins=1, o offset não importa. Se for 4, ele sabe pular os bins sobrepostos.
    num_windows = total_bins - ((window_size_bins - 1) * offset)
    
    # 2. Mapeamento Temporal Dinâmico
    window_centers = []
    window_ends = []
    for w in range(num_windows):
        win_start_time = start_time + (w * step)
        win_end_time = win_start_time + (window_size_bins * bin_width)
        window_centers.append((win_start_time + win_end_time) / 2.0)
        window_ends.append(win_end_time)

    # 3. Definição Dinâmica do Tuning 
    # Em vez de pegar um bloco gigante, pegamos a janela exata que termina em 0ms (ou a mais próxima)
    target_tuning_time = 0 
    
    # Encontra o índice da janela cujo tempo final é o mais próximo de 0ms
    idx_tuning_window = np.argmin(np.abs(np.array(window_ends) - target_tuning_time))
    
    # Extrai estritamente a mesma quantidade de colunas que a janela deslizante vai usar
    offset = bin_width // step
    tuning_bins = [idx_tuning_window + (k * offset) for k in range(window_size_bins)]

    X = df_run_model.drop(columns=['Y'])
    y = df_run_model['Y']
    
    classes = np.unique(y)
    num_classes = len(classes)
    chance_level = (1.0 / num_classes) * 100

    outer_cv = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_accuracies = np.zeros((num_folds, num_windows))
    fold_cms = np.zeros((num_folds, num_windows, num_classes, num_classes))
    fold_f1 = np.zeros((num_folds, num_windows))
    fold_mcc = np.zeros((num_folds, num_windows))
    fold_gm = np.zeros((num_folds, num_windows))


    if verbose:
        print(f"Iniciando Decoding ({num_folds} folds, {num_windows} janelas, {total_bins} bins totais)...")
        print(f"Bins selecionados para Tuning: {tuning_bins}")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        if verbose:
            print(f"--- Processando Fold {fold_idx + 1}/{num_folds} --- {pd.Timestamp.now().strftime('%H:%M:%S')}")
            
        X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Padronização
        X_train_z, X_test_z = zscore_per_neuron(X_train_raw, X_test_raw)
        
        # Tuning Pós-Sacádico Dinâmico
        # Usa o nome exato da coluna para evitar o bug do "endswith" 
        post_cols = [f"{nid}_Bin{b}" for nid in neuron_ids for b in tuning_bins]
        X_train_post = X_train_z[post_cols]
        
        search = RandomizedSearchCV(
            estimator=LinearSVC(class_weight='balanced', dual=False),
            param_distributions={'C': loguniform(1e-4, 1e2)},
            n_iter=15,
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
            scoring='balanced_accuracy',
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train_post, y_train)
        best_C = search.best_params_['C']
        
        # Treino e Avaliação nas Moving Windows
        best_svm = LinearSVC(class_weight='balanced', C=best_C, dual=False, max_iter=2000)
        
        for w_idx in range(num_windows):
            # A mágica do offset: garante que ele pega os bins isolados corretos
            active_bins = [w_idx + (k * offset) for k in range(window_size_bins)]
            win_cols = [f"{nid}_Bin{b}" for nid in neuron_ids for b in active_bins]
            
            X_tr_win = X_train_z[win_cols]
            X_te_win = X_test_z[win_cols]
            
            best_svm.fit(X_tr_win, y_train)
            preds = best_svm.predict(X_te_win)
            
            # Acurácia
            acc = balanced_accuracy_score(y_test, preds)
            fold_accuracies[fold_idx, w_idx] = acc
            
            # Matriz de Confusão Normalizada
            cm = confusion_matrix(y_test, preds, labels=classes, normalize='true')
            fold_cms[fold_idx, w_idx] = cm

            # Suas métricas atuais
            acc = balanced_accuracy_score(y_test, preds)
            fold_accuracies[fold_idx, w_idx] = acc
            
            # NOVAS MÉTRICAS
            fold_f1[fold_idx, w_idx] = f1_score(y_test, preds, average='macro')
            mcc_raw = matthews_corrcoef(y_test, preds)
            fold_mcc[fold_idx, w_idx] = ((mcc_raw + 1) / 2)

            recalls = np.diag(cm)
            fold_gm[fold_idx, w_idx] = np.prod(recalls) ** (1.0 / num_classes)

    mean_acc = np.mean(fold_accuracies, axis=0) * 100
    sem_acc = (np.std(fold_accuracies, axis=0) / np.sqrt(num_folds)) * 100
    mean_cms = np.mean(fold_cms, axis=0) * 100
    mean_f1 = np.mean(fold_f1, axis=0) * 100
    mean_mcc = np.mean(fold_mcc, axis=0) * 100
    mean_gm = np.mean(fold_gm, axis=0) * 100

    return {
        'mean_acc': mean_acc,
        'sem_acc': sem_acc,
        'mean_f1': mean_f1,
        'mean_mcc': mean_mcc,
        'mean_gm':mean_gm,
        'window_centers': np.array(window_centers),
        'window_ends': np.array(window_ends),
        'chance_level': chance_level,
        'fold_accuracies': fold_accuracies * 100,
        'mean_cms': mean_cms,
        'classes': classes
    }